# Этап 3: INT8 Quantization — Полный сравнительный анализ

Мы сравниваем две модели по трем ключевым бизнес-метрикам:
1. **Memory (Память)**: Физический объем занимаемого места.
2. **Throughput (Пропускная способность)**: Сколько символов генерируется в секунду.
3. **Latency (Задержка)**: Время до появления самого первого символа (Time to First Token).

In [10]:
# Настройка путей: если мы в подпапке, переходим в корень проекта

import torch
import copy
import time
import pandas as pd
import numpy as np
from src.model import GPTLanguageModel, device
from src.utils import get_batch, estimate_loss, decode, encode

def get_model_size_mb(mdl, real_int8=False):
    if not real_int8:
        param_size = sum(p.nelement() * p.element_size() for p in mdl.parameters())
        buffer_size = sum(b.nelement() * b.element_size() for b in mdl.buffers())
        return (param_size + buffer_size) / 1024**2
    else:
        total_bits = 0
        for name, param in mdl.named_parameters():
            bits = 8 if ('weight' in name and param.dim() > 1) else 32
            total_bits += param.nelement() * bits
        for b in mdl.buffers():
            total_bits += b.nelement() * 32
        return total_bits / (8 * 1024**2)

@torch.no_grad()
def measure_performance(mdl, num_tokens=50):
    context = torch.zeros((1, 1), dtype=torch.long, device=device)
    
    # Warmup
    _ = mdl.generate(context, max_new_tokens=5)
    
    # 1. Latency (TTFT)
    start_latency = time.time()
    _ = mdl.generate(context, max_new_tokens=1)
    latency = (time.time() - start_latency) * 1000
    
    # 2. Throughput
    start_throughput = time.time()
    _ = mdl.generate(context, max_new_tokens=num_tokens)
    duration = time.time() - start_throughput
    throughput = num_tokens / duration
    
    return latency, throughput

def quantize_tensor_int8(x):
    x_max = x.abs().max().item()
    if x_max == 0: return x
    scale = x_max / 127.0
    return torch.round(x / scale).clamp(-128, 127) * scale

### 1. Замер исходной модели (Baseline FP32)
Здесь мы фиксируем «золотой стандарт» качества и скорости.

In [12]:
model_fp32 = GPTLanguageModel().to(device)
try:
    model_fp32.load_state_dict(torch.load('nanoGPT-lab/model_ckpt.pt', map_location=device))
    print("✅ Успешно загружены веса модели.")
except:
    print("⚠️ Чекпоинт не найден, замеры будут на случайных весах.")
model_fp32.eval()

print("\n--- [STEP 1] Baseline FP32 Performance ---")
loss_fp32 = estimate_loss(model_fp32)['val'].item()
size_fp32 = get_model_size_mb(model_fp32)
lat_fp32, thr_fp32 = measure_performance(model_fp32)

print(f"🔹 Memory Usage:     {size_fp32:.2f} MB")
print(f"🔹 Inference Latency: {lat_fp32:.2f} ms (Time to First Token)")
print(f"🔹 Throughput Rate:   {thr_fp32:.2f} tokens/sec")
print(f"🔹 Model Quality:     {loss_fp32:.4f} (Val Loss)")

✅ Успешно загружены веса модели.

--- [STEP 1] Baseline FP32 Performance ---
🔹 Memory Usage:     50.16 MB
🔹 Inference Latency: 10.50 ms (Time to First Token)
🔹 Throughput Rate:   70.28 tokens/sec
🔹 Model Quality:     1.5019 (Val Loss)


In [13]:
from src.utils import get_batch

try:
    model_fp32.load_state_dict(torch.load('nanoGPT-lab/model_ckpt.pt', map_location=device))
    print("✅ Загружены обученные веса из model_ckpt.pt")
except:
    print("⚠️ Чекпоинт не найден, используем случайные веса (результаты будут менее точными)")

model.eval()

def measure_sensitivity(mdl, noise_level=0.01):
    xb, yb = get_batch('val')
    
    # Базовый лосс
    with torch.no_grad():
        _, base_loss = mdl(xb, yb)
    
    print(f"Базовый Loss на валидации: {base_loss.item():.4f}\n")
    print(f"{'Layer to noise':<40} | {'Delta Loss':<10} | {'Sensitivity'}")
    print("-" * 75)

    for name, param in mdl.named_parameters():
        if 'weight' in name and param.dim() > 1:
            orig_data = param.data.clone()
            
            # Добавляем шум
            noise = torch.randn_like(param.data) * noise_level
            param.data.add_(noise)
            
            with torch.no_grad():
                _, new_loss = mdl(xb, yb)
            
            delta = new_loss.item() - base_loss.item()
            
            # Оценка: чем выше Sensitivity, тем опаснее сжимать этот слой
            sensitivity = "🔴 HIGH" if delta > 0.05 else ("🟡 MED" if delta > 0.01 else "🟢 LOW")
            
            print(f"{name[:40]:<40} | {delta:<10.4f} | {sensitivity}")
            
            # Возвращаем веса назад!
            param.data.copy_(orig_data)

measure_sensitivity(model_fp32)

✅ Загружены обученные веса из model_ckpt.pt
Базовый Loss на валидации: 1.5011

Layer to noise                           | Delta Loss | Sensitivity
---------------------------------------------------------------------------
token_embedding_table.weight             | 0.0399     | 🟡 MED
position_embedding_table.weight          | 0.0286     | 🟡 MED
blocks.0.sa.heads.0.key.weight           | 0.0035     | 🟢 LOW
blocks.0.sa.heads.0.query.weight         | -0.0004    | 🟢 LOW
blocks.0.sa.heads.0.value.weight         | 0.0060     | 🟢 LOW
blocks.0.sa.heads.1.key.weight           | 0.0065     | 🟢 LOW
blocks.0.sa.heads.1.query.weight         | 0.0041     | 🟢 LOW
blocks.0.sa.heads.1.value.weight         | 0.0031     | 🟢 LOW
blocks.0.sa.heads.2.key.weight           | 0.0088     | 🟢 LOW
blocks.0.sa.heads.2.query.weight         | 0.0066     | 🟢 LOW
blocks.0.sa.heads.2.value.weight         | 0.0098     | 🟢 LOW
blocks.0.sa.heads.3.key.weight           | 0.0059     | 🟢 LOW
blocks.0.sa.heads.3.query.weight 

### 2. Квантование и замер INT8
Теперь мы «портим» веса округлением до 8 бит и смотрим, как изменятся те же метрики.

In [ ]:
# Функция для эксперимента
def run_quantization_experiment(model_fp32, sensitive_layers=[], title="INT8"):
    print(f"\n--- {title} Performance ---")
    if sensitive_layers:
        print(f"Skipping sensitive layers: {sensitive_layers}")
    
    model_q = copy.deepcopy(model_fp32)
    with torch.no_grad():
        for name, param in model_q.named_parameters():
            if 'weight' in name and param.dim() > 1:
                if name not in sensitive_layers:
                    param.copy_(quantize_tensor_int8(param.data))
    
    loss = estimate_loss(model_q)['val'].item()
    size = get_model_size_mb(model_q, real_int8=True)
    lat, thr = measure_performance(model_q)
    
    print(f"🔸 Memory:     {size:.2f} MB")
    print(f"🔸 Latency:    {lat:.2f} ms")
    print(f"🔸 Throughput: {thr:.2f} tok/s")
    print(f"🔸 Quality:    {loss:.4f} (Val Loss)")
    
    return model_q, size, lat, thr, loss

# 1. Full INT8 (Без ограничений)
print("Running Full INT8 Quantization...")
_, size_full_int8, lat_full_int8, thr_full_int8, loss_full_int8 = run_quantization_experiment(
    model_fp32, 
    sensitive_layers=[], 
    title="[STEP 2a] Full INT8"
)

# 2. Mixed Precision (С учетом чувствительных слоев)
print("\nRunning Mixed Precision Quantization...")
sensitive_layers = ['token_embedding_table.weight', 'position_embedding_table.weight', 'lm_head.weight']
model_int8, size_int8, lat_int8, thr_int8, loss_int8 = run_quantization_experiment(
    model_fp32, 
    sensitive_layers=sensitive_layers, 
    title="[STEP 2b] Mixed Precision (Sensitive Layers FP32)"
)


--- [STEP 2] Mixed Precision (INT8 + FP32) Performance ---
🔸 Memory Usage:     19.36 MB (Estimated storage size)
🔸 Inference Latency: 10.61 ms
🔸 Throughput Rate:   70.54 tokens/sec
🔸 Model Quality:     1.5039 (Val Loss)


### 3. Анализ эффективности
Сравним выигрыш в ресурсах против потери качества.

In [21]:
results = {
    "Metric": ["Memory (MB)", "Throughput (tokens/s)", "Latency (ms)", "Validation Loss"],
    "FP32": [size_fp32, thr_fp32, lat_fp32, loss_fp32],
    "Full INT8": [size_full_int8, thr_full_int8, lat_full_int8, loss_full_int8],
    "Mixed INT8": [size_int8, thr_int8, lat_int8, loss_int8],
    "Delta (Mixed)": [
        f"{size_fp32/size_int8:.1f}x smaller", 
        f"{(thr_int8/thr_fp32 - 1)*100:+.1f}% check", 
        f"{lat_int8 - lat_fp32:+.2f} ms", 
        f"{(loss_int8/loss_fp32 - 1)*100:+.2f}% quality loss"
    ]
}

df = pd.DataFrame(results)
display(df)

# Сравниваем потери качества
quality_drop_full = (loss_full_int8 / loss_fp32 - 1) * 100
quality_drop_mixed = (loss_int8 / loss_fp32 - 1) * 100

print(f"\n📉 Потеря качества (Full INT8): {quality_drop_full:+.2f}%")
print(f"📉 Потеря качества (Mixed INT8): {quality_drop_mixed:+.2f}%")

if quality_drop_mixed < quality_drop_full:
    print("✅ Mixed Precision лучше сохранил качество модели!")
else:
    print("ℹ️ Разница в качестве незначительна.")

,Metric,FP32,Full INT8,Mixed INT8,Delta (Mixed)
0,Memory (MB),50.156498,19.357426,19.357426,2.6x smaller
1,Throughput (tokens/s),70.284344,68.566040,71.587077,+1.9% check
2,Latency (ms),10.499954,10.828972,9.872913,-0.63 ms
3,Validation Loss,1.501920,1.504737,1.500315,-0.11% quality loss



📉 Потеря качества (Full INT8): +0.19%
📉 Потеря качества (Mixed INT8): -0.11%
✅ Mixed Precision лучше сохранил качество модели!


### 4. Генерация текста (Blind Test)
Напишем промпт и посмотрим разницу в стиле. 
*Если вы видите NameError, убедитесь, что выполнили самую первую ячейку кода.*

In [26]:
from src.utils import decode, encode

prompt = "ROMEO: "
context = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)

print("--- FP32 OUTPUT ---")
print(decode(model_fp32.generate(context, max_new_tokens=1000)[0].tolist()))

print("\n--- INT8 OUTPUT ---")
print(decode(model_int8.generate(context, max_new_tokens=1000)[0].tolist()))

--- FP32 OUTPUT ---
ROMEO: pier, for his girth; and make him spitest too.

HENRY PEYmiBaBROKE:
And, this, how! my queen, fortune! thy nurse!
O thou oft, hear!
Thou art evilousance! was nothing past, spring and his George,
And henceth a storm livour'd to serve and with
And Neckle against my fair Thursday:
I this thy dead,
In bringed me, should blood in his undance,
And fear in his facem cleing than advise against and man
dies little.
Ah, good blood, a wordder, his leas
May a place, are you soons; geld my loving one
pain in was the accusant from head. I talk her, she
that you have show'd meet's lent againstrike, butches-sent prese
doors, thou mays the beauty of you from one care
helds me but seeing.

WERUTgards,
You shall buried her her arm;
That I would be gone, to your king, right:
what, which, and what indeed is Clifflius shall;
And yet in again day?

DUKE OF YORK:
Sort tell myself is that dog. I'll not me hath watcher
Fourton in deers; yet be refused;
May with your tenor liberty end 

In [ ]:
torch.save(model_int8.state_dict(), 'nanoGPT-lab/model_int8.pt')
print("✅ Quantized model saved to model_int8.pt")

✅ Quantized model saved to model_int8.pt
